In [1]:
import scipy.io as sio
import numpy as np
import pandas as pd
import time

np.seterr(all='ignore')
SEED = 62
N = 517
MAT_FILE = '870d37a7-673d-49ff-8a98-47b9b519237f.mat'

RULES = ['Zimmermann1','Zimmermann2','Luhandjula','Werners1','Werners2',
         'Sales1','Sales2','Mizumoto1','Mizumoto2','R5p']
PVALS = np.round(np.arange(0.0, 1.05, 0.05), 2)

# ── Missing-modality substitution strategies (no noise) ────────────────────
#   'zero'   : fill with 0.0  (worst quality)
#   'neutral': fill with 0.5  (maximum ignorance)
#   'unit'   : fill with 1.0  (best quality)
#   'skip'   : drop missing modality; fuse only remaining ones
FILL_STRATEGIES = {'zero': 0.0, 'neutral': 0.5, 'unit': 1.0, 'skip': None}

d = sio.loadmat(MAT_FILE)
raw_keys = {
    'Face_C':    'fingToFaceCSCORES', 'Face_G':    'fingToFaceGSCORES',
    'Finger_LI': 'VfingToFaceliSCORES', 'Finger_RI': 'VfingToFaceriSCORES',
}
rng = np.random.default_rng(SEED)
perm = rng.permutation(N)
folds = np.array_split(perm, 5)


def minmax_fit_transform(S_full, train_ids, test_ids):
    S_full = S_full.astype(float)
    S_train = S_full[np.ix_(train_ids, train_ids)]
    S_test  = S_full[np.ix_(test_ids, test_ids)]
    invalid = (S_train == -1)
    valid_vals = S_train[~invalid]
    vmin, vmax = valid_vals.min(), valid_vals.max()
    def transform(S):
        inv = (S == -1)
        out = np.full_like(S, -1.0)
        out[~inv] = (S[~inv] - vmin) / (vmax - vmin)
        return out
    return transform(S_train), transform(S_test)


def mask_modality(S_clean, fill_value):
    """Set all valid scores to fill_value; invalid entries remain -1."""
    invalid = (S_clean == -1)
    S_masked = S_clean.copy()
    S_masked[~invalid] = fill_value
    return S_masked


def apply_rule(rule, x, y, p):
    valid = (x != -1) & (y != -1)
    xv = np.where(valid, x, 0.0); yv = np.where(valid, y, 0.0)
    with np.errstate(invalid='ignore', divide='ignore'):
        if rule == 'Zimmermann1':
            xy = xv*yv; s = xv+yv-xy
            xy_c = np.clip(xy, 0, None); s_c = np.clip(s, 0, None)
            term1 = np.power(xy_c, 1-p, where=(xy_c>0), out=np.zeros_like(xy_c))
            term1 = np.where(xy_c==0, 0.0**(1-p) if (1-p)!=0 else 1.0, term1)
            term2 = np.power(s_c, p, where=(s_c>0), out=np.zeros_like(s_c))
            term2 = np.where(s_c==0, 0.0**p if p!=0 else 1.0, term2)
            F = term1*term2
        elif rule == 'Zimmermann2': F = np.minimum(xv,yv)*(1-p) + np.maximum(xv,yv)*p
        elif rule == 'Luhandjula': F = np.minimum(xv,yv)*(1-p) + np.minimum(1,xv+yv)*p
        elif rule == 'Werners1': F = np.minimum(xv,yv)*(1-p) + ((xv+yv)/2)*p
        elif rule == 'Werners2': F = np.maximum(xv,yv)*(1-p) + ((xv+yv)/2)*p
        elif rule == 'Sales1': F = np.maximum(0,xv+yv-1)*(1-p) + np.minimum(xv,yv)*p
        elif rule == 'Sales2': F = np.minimum(1,xv+yv)*(1-p) + np.maximum(xv,yv)*p
        elif rule == 'Mizumoto1': F = np.minimum(xv,yv)*(1-p) + np.sqrt(np.clip(xv*yv,0,None))*p
        elif rule == 'Mizumoto2': F = ((xv+yv)/2)*(1-p) + np.sqrt(np.clip(xv*yv,0,None))*p
        elif rule == 'R5p':
            s = xv+yv-xv*yv
            F = p*s + (1-p)*(1-s)*np.minimum(xv,yv)
        else: raise ValueError(rule)
    F = np.where(valid, F, -1.0)
    F = np.clip(F, np.where(valid,0,-1), 1)
    return F


# ── System definitions (no noise injection) ───────────────────────────────
SYSTEM_DEFS = {
    'S1': dict(stage1=('Face_C','Face_G'),         stage2=None,    cascaded=False),
    'S2': dict(stage1=('Face_C','Finger_LI'),       stage2=None,    cascaded=False),
    'S3': dict(stage1=('Face_C','Finger_RI'),       stage2=None,    cascaded=False),
    'S4': dict(stage1=('Face_G','Finger_LI'),       stage2=None,    cascaded=False),
    'S5': dict(stage1=('Face_G','Finger_RI'),       stage2=None,    cascaded=False),
    'S6': dict(stage1=('Finger_LI','Finger_RI'),    stage2=None,    cascaded=False),
    'S7': dict(stage1=('Finger_LI','Finger_RI'),    stage2='Face_C', cascaded=True),
    'S8': dict(stage1=('Finger_LI','Finger_RI'),    stage2='Face_G', cascaded=True),
}
SYSTEMS = list(SYSTEM_DEFS.keys())


def get_modalities(system):
    """Return all modalities used by a system."""
    sdef = SYSTEM_DEFS[system]
    mods = list(sdef['stage1'])
    if sdef['cascaded'] and sdef['stage2']:
        mods.append(sdef['stage2'])
    return mods


def eer_fast(genuine, impostor):
    if len(genuine)==0 or len(impostor)==0: return np.nan
    G = np.sort(genuine); I = np.sort(impostor)
    thr = np.unique(np.concatenate([G,I]))
    far = 1 - np.searchsorted(I, thr, side='left')/len(I)
    frr = np.searchsorted(G, thr, side='left')/len(G)
    diff = np.abs(far-frr)
    idx = np.argmin(diff)
    return (far[idx]+frr[idx])/2.0


def rank1_at_far(S, far_target=0.01):
    n_ = S.shape[0]; diag_idx = np.arange(n_)
    diag = np.diag(S); off_mask = ~np.eye(n_, dtype=bool)
    impostor = S[off_mask]; impostor_valid = impostor[impostor != -1]
    if len(impostor_valid)==0: return np.nan
    t = np.percentile(impostor_valid, 100*(1-far_target))
    M = np.where(S==-1, -np.inf, S)
    top_score = M.max(axis=1); top_idx = M.argmax(axis=1)
    valid_probe = diag != -1
    correct = (top_idx == diag_idx); accept = (top_score >= t)
    hit = correct & accept & valid_probe
    n_probes = valid_probe.sum()
    return hit.sum()/n_probes if n_probes else np.nan


def genuine_impostor(S):
    n_ = S.shape[0]
    diag = np.diag(S); off_mask = ~np.eye(n_, dtype=bool)
    genuine = diag[diag != -1]
    imp_all = S[off_mask]; impostor = imp_all[imp_all != -1]
    return genuine, impostor


def build_matrix_baseline(system, mods_clean, rule, p):
    """All modalities present (no missing)."""
    sdef = SYSTEM_DEFS[system]
    a, b = sdef['stage1']
    fused = apply_rule(rule, mods_clean[a], mods_clean[b], p)
    if sdef['cascaded']:
        fused = apply_rule(rule, fused, mods_clean[sdef['stage2']], p)
    return fused


def build_matrix_missing(system, mods_clean, missing_mod, fill_value, rule, p):
    """One modality is missing and replaced with fill_value (zero / neutral / unit)."""
    sdef = SYSTEM_DEFS[system]
    a, b = sdef['stage1']

    xa = mask_modality(mods_clean[a], fill_value) if missing_mod == a else mods_clean[a]
    xb = mask_modality(mods_clean[b], fill_value) if missing_mod == b else mods_clean[b]

    fused = apply_rule(rule, xa, xb, p)
    if sdef['cascaded']:
        third = sdef['stage2']
        x3 = mask_modality(mods_clean[third], fill_value) if missing_mod == third else mods_clean[third]
        fused = apply_rule(rule, fused, x3, p)
    return fused


def build_matrix_skip(system, mods_clean, missing_mod, rule, p):
    """Drop the missing modality; fuse only the remaining ones (identity for single cue)."""
    sdef = SYSTEM_DEFS[system]
    all_mods = get_modalities(system)
    available = [m for m in all_mods if m != missing_mod]

    if len(available) == 0:
        return np.full_like(mods_clean[all_mods[0]], -1.0)
    if len(available) == 1:
        return mods_clean[available[0]].copy()   # single modality → identity
    # Two available: fuse them
    return apply_rule(rule, mods_clean[available[0]], mods_clean[available[1]], p)


# ═══════════════════════════════════════════════════════════════════════════
#  Main loop — missing modality (no noise)
# ═══════════════════════════════════════════════════════════════════════════
t0 = time.time()
rows = []

for fold_idx in range(5):
    test_ids = np.sort(folds[fold_idx])
    train_ids = np.sort(np.setdiff1d(np.arange(N), test_ids))

    clean_test = {}
    for name, key in raw_keys.items():
        _, Ste = minmax_fit_transform(d[key], train_ids, test_ids)
        clean_test[name] = Ste

    for system in SYSTEMS:
        system_mods = get_modalities(system)

        # ── Baseline: all modalities present ──────────────────────────────
        for rule in RULES:
            for p in PVALS:
                F = build_matrix_baseline(system, clean_test, rule, p)
                genuine, impostor = genuine_impostor(F)
                rows.append(dict(
                    Fold=fold_idx+1, System=system,
                    Missing='none', Fill='baseline',
                    Rule=rule, p=p,
                    EER=eer_fast(genuine, impostor),
                    Rank1=rank1_at_far(F, 0.01)
                ))

        # ── Missing-modality scenarios ────────────────────────────────────
        for missing_mod in system_mods:
            for fill_name, fill_value in FILL_STRATEGIES.items():
                for rule in RULES:
                    for p in PVALS:
                        if fill_name == 'skip':
                            F = build_matrix_skip(system, clean_test, missing_mod, rule, p)
                        else:
                            F = build_matrix_missing(
                                system, clean_test, missing_mod, fill_value, rule, p)
                        genuine, impostor = genuine_impostor(F)
                        rows.append(dict(
                            Fold=fold_idx+1, System=system,
                            Missing=missing_mod, Fill=fill_name,
                            Rule=rule, p=p,
                            EER=eer_fast(genuine, impostor),
                            Rank1=rank1_at_far(F, 0.01)
                        ))

    print(f'Fold {fold_idx+1}/5 done, elapsed {round(time.time()-t0,1)}s')

df = pd.DataFrame(rows)
df.to_excel('./Results/missing_modality_raw.xlsx', index=False)
print(df.shape)
print("DONE")

Fold 1/5 done, elapsed 34.8s
Fold 2/5 done, elapsed 82.3s
Fold 3/5 done, elapsed 125.8s
Fold 4/5 done, elapsed 168.8s
Fold 5/5 done, elapsed 213.0s
(84000, 8)
DONE
